# سبق 18: AI ایجنٹس کو کرپٹوگرافک رسیدوں کے ساتھ محفوظ بنانا

## عملی نوٹ بک

یہ نوٹ بک چار مشقوں کے ذریعے گزرے گا:

1. **اپنی پہلی رسید پر دستخط کریں** ایک ایجنٹ ٹول کال کے لیے اور اسے تصدیق کریں۔
2. **رسید میں چھیڑ چھاڑ کریں** اور تصدیق کے ناکام ہونے کو دیکھیں۔
3. **ایک تین رسیدی چین بنائیں** اور چین کی سالمیت کی تصدیق کریں۔
4. **مائیکروسافٹ ایجنٹ فریم ورک ٹول کال کو لپیٹیں** تاکہ ہر کارروائی ایک رسید جاری کرے۔

تمام کرپٹوگرافک پرائمریٹو کو اچھی طرح برقرار رکھی جانے والی لائبریریوں سے درآمد کیا گیا ہے (`pynacl` Ed25519 کے لیے، `jcs` RFC 8785 کینونیکل JSON کے لیے، `hashlib` پائتھون کی معیاری لائبریری سے SHA-256 کے لیے)۔ رسید کا منطق خود سادہ پائتھون ہے جسے آپ پڑھ اور ترمیم کر سکتے ہیں۔

سیلز کو ترتیب سے چلائیں۔ ہر سیکشن مختصر اور خود مختار ہے۔


## سیٹ اپ

دونوں انحصارات انسٹال کریں۔ دونوں کے اجازت نامے (Apache-2.0 / MIT) نرم ہیں۔


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## مددگار معاونت

یہ دو معاونز base64url انکوڈنگ (بغیر پیڈنگ کے) اور بے ترتیب اشیاء کے SHA-256 ہیشنگ کو سنبھالتے ہیں۔ یہ نوٹ بک کے باقی حصے کو رسید کی منطق پر مرکوز رکھتے ہیں۔


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## سیکشن 1: اپنا پہلا رسید دستخط کریں

تصور کریں کہ ہمارے ایجنٹ **Contoso Travel** کا ابھی ایک گاہک کے لئے سڈنی سے لاس اینجلس کے لیے پروازیں دیکھ رہا ہے۔ ہم اس ٹول کال کو ایک دستخط شدہ رسید کے طور پر ریکارڈ کرنا چاہتے ہیں تاکہ مستقبل میں کوئی آڈیٹر اس کی تصدیق کر سکے بغیر ہم پر اعتماد کیے۔

### مرحلہ 1.1: دستخط کرنے کی کلید بنائیں

پیداواری ماحول میں، ایجنٹ کی دستخط کرنے والی کلید ہارڈویئر سیکیورٹی ماڈیول (HSM)، Azure Key Vault، یا کسی ملتے جلتے محفوظ اسٹور میں محفوظ رہے گی۔ اس سبق کے لئے ہم میموری میں ایک نئی کلید تیار کرتے ہیں۔


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### مرحلہ 1.2: رسید کا پیلوڈ تیار کریں

پیلوڈ میں وہ سب کچھ شامل ہوتا ہے جس کی ہم رسید سے تصدیق چاہتے ہیں: کس نے عمل کیا، کون سا آلہ استعمال کیا، کن دلائل کے ساتھ، کیا نتیجہ واپس آیا، کس پالیسی کے تحت، اور کب۔ ہم دلائل اور نتیجے کو شامل کرنے کے بجائے ہیش کرتے ہیں تاکہ رسید حساس مواد کو افشاء نہ کرے۔


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### مرحلہ 1.3: رسید پر دستخط کریں اور اسے جمع کریں

تین مراحل:

1. پے لوڈ کو JCS کے ذریعے کینونیکلائز کریں تاکہ دو مختلف نفاذات جو ایک ہی منطقی رسید بناتے ہیں وہ بائٹ لحاظ سے بالکل ایک جیسے ہوں۔
2. کینونیکل بائٹس کو SHA-256 سے ہیش کریں۔
3. ہیش پر Ed25519 پرائیوٹ کلید سے دستخط کریں۔

دستخط پھر اصل پے لوڈ کے ساتھ منسلک کیا جاتا ہے تاکہ آخری رسید تیار کی جا سکے۔


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### مرحلہ 1.4: رسید کی تصدیق کریں

تصدیق عمل کو الٹا کرتی ہے۔ ہم دستخط کو ہٹا دیتے ہیں، کینونیکل ہیش کو دوبارہ حساب کرتے ہیں، اور رسید میں موجود عوامی کلید کے خلاف دستخط کی جانچ کرتے ہیں۔

ایک آڈیٹر جو یہ تصدیق کر رہا ہے، اسے ہم سے کچھ بھی نہیں چاہیے سوائے خود رسید کے۔ کوئی سروس کال کرنے کی ضرورت نہیں، کوئی کلید کی ڈائریکٹری سے استفسار نہیں، کوئی اعتماد درکار نہیں۔


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

آپ کو `Receipt is valid: True` نظر آنا چاہیے۔ ایجنٹ نے اپنا پہلا رمز نگاری شدہ دستخط شدہ آڈٹ ریکارڈ تیار کر لیا ہے۔


## سیکشن 2: رسید میں چھیڑچھاڑ کریں

رسیدوں کا پورا مقصد یہ ہے کہ وہ چھیڑچھاڑ کا پتا دیتی ہیں۔ آئیے اسے ثابت کرتے ہیں۔

ہم رسید کے بالکل ایک حرف کو تبدیل کریں گے اور ویری فکیشن کا ناکام ہونا دیکھیں گے۔


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ابھی کیا ہوا؟

جب ہم نے `policy_id` کو تبدیل کیا، تو کینونیکل بائٹس تبدیل ہو گئیں۔ ان بائٹس کا SHA-256 ہیش بدل گیا۔ دستخط (جو اصل ہیش پر تھا) اب نئے ہیش سے ملتا نہیں ہے۔ تصدیق صحیح طور پر `False` لوٹاتی ہے۔

رسید کے کسی بھی فیلڈ کو تبدیل کر کے اسے تصدیق کرانا ممکن نہیں، جب تک کہ حملہ آور کے پاس پرائیویٹ کی نہ ہو۔ جب تک پرائیویٹ کی کی کی والٹ میں ہے اور پبلک کی شائع کی گئی ہے، چھیڑ چھاڑ چھپانا ناممکن ہے۔

خود آزما کر دیکھیں: اوپر دی گئی سیل میں `tool_name` یا `agent_id` یا `timestamp` کو تبدیل کریں اور دوبارہ چلائیں۔ ہر تبدیلی ایک ناقابل قبول رسید پیدا کرتی ہے۔


## سیکشن 3: رسیدوں کو ایک ساتھ جوڑنا

ایک واحد رسید ایک عمل کو محفوظ رکھتی ہے۔ زیادہ تر ایجنٹ کئی اعمال کرتے ہیں۔ پورے سلسلے کو چھیڑ چھاڑ سے بچانے کے لیے، ہم ہر رسید کو پچھلی رسید کے ہیش کو نئی رسید کے پیلوڈ میں شامل کرکے پچھلی رسید سے جوڑتے ہیں۔

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

اگر کوئی رسید کو ہٹا دیتا ہے یا ترتیب بدل دیتا ہے، تو سلسلہ بالکل اسی نقطے پر ٹوٹ جاتا ہے۔ کسی بھی بعد کی رسید کی تصدیق ناکام ہو جاتی ہے کیونکہ اس کا `previous_receipt_hash` اب اس کے پیش رو کے اصل ہیش سے میل نہیں کھاتا۔


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

اب زنجیر کو توڑیں درمیانی رسید میں چھڑ چھاڑ کرکے اور دوبارہ تصدیق کریں۔ چھڑی ہوئی رسید اپنی دستخط کی جانچ میں ناکام ہو جاتی ہے، اور اگلی رسید اپنی زنجیر-لنک کی جانچ میں ناکام ہو جاتی ہے (کیونکہ اس کا `previous_receipt_hash` اب ترمیم شدہ درمیانی رسید کے ہیش سے میل نہیں کھاتا)۔


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

رسید 0 اب بھی تصدیق کرتی ہے (یہ تبدیل نہیں ہوئی اور اس کا کوئی پیش رو جس پر انحصار کیا جا سکے موجود نہیں ہے)۔ رسید 1 اپنی دستخط کی جانچ میں ناکام ہو جاتی ہے کیونکہ ہم نے `tool_args_hash` کو تبدیل کیا ہے۔ رسید 2 اپنی چِین-لنک چیک میں ناکام ہو جاتی ہے کیونکہ اس کا `previous_receipt_hash` اصل (اب تبدیل شدہ) رسید 1 کے خلاف حساب کیا گیا تھا۔

اگرچہ ایک حملہ آور تبدیل شدہ رسید 1 پر دوبارہ دستخط کر دیتا ہے (جو وہ خفیہ کلید کے بغیر نہیں کر سکتے)، رسید 2 میں چین-لنک کی مطابقت نہ ہونے کی وجہ سے چالاکی ظاہر ہو جائے گی۔ تبدیلی کو چھپانے کے لیے، حملہ آور کو اس تبدیلی کے بعد ہر رسید پر دوبارہ دستخط کرنا ہوں گے، جس کے لیے خفیہ کلید کا مالک ہونا ضروری ہے۔


## سیکشن 4: ایجنٹ ٹول کال کو رسید دستخط کے ساتھ لپیٹنا

ایک حقیقی تعیناتی میں، آپ نہیں چاہتے کہ ہر ایجنٹ مصنف `make_receipt` کو یاد رکھے کال کرنا۔ آپ چاہتے ہیں کہ ہر ٹول کال کے لیے رسید دستخط خودکار ہو۔

یہاں سب سے آسان نمونہ ہے: ایک ریپر کلاس جو کسی بھی کال ایبل ٹول فنکشن کو لیتا ہے اور اس کا ایک رسید جاری کرنے والا ورژن واپس دیتا ہے۔ یہ کسی بھی ایجنٹ فریم ورک کے لیے مطابقت رکھتا ہے، بشمول مائیکروسافٹ ایجنٹ فریم ورک (`agent_framework.foundry`)۔

اگر آپ کے پاس مائیکروسافٹ فاؤنڈری پروجیکٹ سیٹ اپ نہیں ہے، تو نیچے دیا گیا مقامی موک پھر بھی اس نمونے کی وضاحت کرتا ہے۔


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### مائیکروسافٹ ایجنٹ فریم ورک کے ساتھ انضمام  

اوپر دیا گیا `ReceiptedTool` ریپر فریم ورک سے آزاد ہے۔ اسے مائیکروسافٹ ایجنٹ فریم ورک کے ساتھ بنائے گئے ایجنٹ کے اندر استعمال کرنے کے لیے، ریپ کیے ہوئے فنکشن کو ایک ٹول کے طور پر رجسٹر کریں۔ ایک خاکہ (آپ اس نقلی کو اصلی مائیکروسافٹ فانڈری ٹول رجسٹریشن سے تبدیل کریں گے):  

```python
# انضمام کی شکل دکھانے والا پسیوڈوکوڈ۔
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="آپ کنٹوسو ٹریول ایجنٹ ہیں ...",
#     tools=[receipted_lookup],   # لپٹا ہوا ٹول، نہ کہ خام فنکشن
# )
# response = agent.run("جون میں سڈنی سے لاس اینجلس کے لیے پروازیں تلاش کریں۔")
#
# # چلانے کے بعد، ہر ٹول کال جسے ایجنٹ نے کیا ہے، اس کا ایک دستخط شدہ رسید ہوتا ہے:
# audit_chain = receipted_lookup.receipts
```
  
ایجنٹ فریم ورک کو رسیدوں کے بارے میں کچھ جاننے کی ضرورت نہیں ہے۔ رسید کی دستخط ٹول کے گرد لپٹی ہوئی ہے، فریم ورک میں جوڑی ہوئی نہیں۔ یہ وہ طریقہ ہے جس سے آپ بغیر ایجنٹ کو دوبارہ لکھے موجودہ ایجنٹ کوڈ میں ماخذ کا اضافہ کرتے ہیں۔  


## نتیجہ جات اور توسیعی چیلنج

آپ نے:

- ایک Ed25519 کی جوڑی تیار کی ہے۔
- ایک ایجنٹ ٹول کال کے لیے رسید بنائی اور دستخط کیے ہیں۔
- رسید کو صرف عوامی کلید استعمال کرتے ہوئے آف لائن تصدیق کیا ہے۔
- رسید میں چھیڑ چھاڑ کی اور تصدیق کی ناکامی دیکھی ہے۔
- تین رسیدوں کی ہیش چین والا سلسلہ بنایا ہے۔
- چین کے وسط میں چھیڑ چھاڑ کی اور دستخط کی ناکامی اور چین-لنک ناکامی دونوں دیکھی ہیں۔
- ایک ٹول فنکشن کو خودکار رسید دستخط کے ساتھ لپیٹا ہے۔

**توسیعی چیلنج۔** رسید کے اسکیما میں `request_id` کا فیلڈ شامل کریں (جو تقسیم شدہ ٹریسنگ کے لیے UUID ہے)۔ `make_receipt` کو اپ ڈیٹ کریں تاکہ یہ شامل ہو، اور تصدیق کریں کہ رسیدیں پورے عمل میں ابھی بھی درست ہیں۔ پھر دستخط کے بعد اس فیلڈ میں ترمیم کریں اور تصدیق کی ناکامی کی تصدیق کریں۔ یہ آپ کو یہ سمجھنے پر مجبور کرتا ہے کہ canonical encoding کے ہر بائٹ کا دستخط میں کیا کردار ہے۔

**اہم حد بندی۔** رسیدیں تین چیزیں ثابت کرتی ہیں اور صرف تین چیزیں: نسبت (یہ کلید اس مواد پر دستخط کر چکی ہے)، سالمیت (مواد دستخط کے بعد سے تبدیل نہیں ہوا)، اور ترتیب (یہ رسید اس رسید کے بعد آئی ہے)۔ یہ ثابت نہیں کرتی کہ ایجنٹ کی کارروائی درست تھی، کہ `policy_id` میں نامہ پالیسی واقعی کا جائزہ لیا گیا، یا کہ ایجنٹ نے ہر قاعدہ کی پیروی کی۔ رسیدیں ایک بنیاد ہیں۔ حکمرانی وہ نظام ہے جو آپ اس بنیاد پر بناتے ہیں۔

سبق کا ریڈ می دوبارہ اس حد بندی کو ذہن میں رکھ کر پڑھیں۔ رسیدوں کے ساتھ سب سے عام غلطی یہ سمجھنا ہے کہ "ہمارے پاس رسیدیں ہیں" مطلب "ہم حکومت شدہ ہیں۔" ایسا نہیں ہے۔ رسیدیں ایجنٹ کے رویے کو قابلِ جانچ بناتی ہیں۔ وہ اسے درست نہیں بناتیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
